1: Load Environment & API Key

In [46]:
import os
from dotenv import load_dotenv

# Membaca file .env yang sama
load_dotenv()

# Cek apakah API Key sudah terbaca dengan benar
if os.getenv("GOOGLE_API_KEY"):
    print("✅ API Key Gemini berhasil dimuat!")
else:
    print("❌ API Key BELUM terbaca. Pastikan file .env berada satu folder dengan file .ipynb ini.")

✅ API Key Gemini berhasil dimuat!


2: Load Dokumen PDF

In [47]:
from langchain_community.document_loaders import PyPDFLoader

# Membaca file dokumen.pdf yang sudah kamu download tadi
loader = PyPDFLoader("Esai_Untuk_Indonesia.pdf")
docs = loader.load()

print(f"✅ Berhasil membaca PDF! Total halaman: {len(docs)}")

✅ Berhasil membaca PDF! Total halaman: 2


3: Potong Teks (Text Splitting)

In [48]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)

print(f"✅ Teks berhasil dipecah menjadi {len(splits)} bagian kecil.")

✅ Teks berhasil dipecah menjadi 4 bagian kecil.


In [49]:
!pip install sentence-transformers

4: Inisialisasi Embedding & Simpan ke Chroma DB

In [50]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Embedding alternatif gratis yang berjalan lokal di laptop tanpa API Key Google
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Simpan ke Chroma
vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)
retriever = vectorstore.as_retriever()

print("✅ Vectorstore Chroma dan Embeddings sukses dibuat secara lokal!")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3926.93it/s]


✅ Vectorstore Chroma dan Embeddings sukses dibuat secara lokal!


In [51]:
!pip install langchain-community

In [52]:
!pip install -U langchainhub

In [53]:
!pip install langchain-openai

   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.4 MB ? eta -:--:--
   ----------------------- ---------------- 0.8/1.4 MB 2.6 MB/s eta 0:00:01
   ------------------------------ --------- 1.0/1.4 MB 2.4 MB/s eta 0:00:01
   ---------------------------------------- 1.4/1.4 MB 2.1 MB/s  0:00:00
   ---------------------------------------- 0.0/874.8 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/874.8 kB ? eta -:--:--
   ----------------------------------- ---- 786.4/874.8 kB 2.0 MB/s eta 0:00:01
   ---------------------------------------- 874.8/874.8 kB 2.0 MB/s  0:00:00

   ---------- ----------------------------- 1/4 [tiktoken]
   -------------------- ------------------- 2/4 [openai]
   -------------------- ------------------- 2/4 [openai]
   -------------------- ------------------- 2/4 [openai]
   -------------------- ------------------- 2/4 [openai]
   -------------------- ------------------- 2/4 

5: Jalankan RAG dengan Gemini AI (Output Akhir)

In [ ]:
import time
import os
import wandb
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
from langchain_community.llms import HuggingFaceEndpoint
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. Nonaktifkan W&B sementara untuk testing cepat
os.environ["WANDB_MODE"] = "disabled"
run = wandb.init(project="Generative-AI-RAG", name="RAG_6_Models_Benchmark", mode="disabled")  # Mode disabled untuk testing lokal

# 2. Inisialisasi Model yang AKTIF
llm_gemini_n = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.3)
llm_gemini_c = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.7)

# Menggunakan Model Mini Terbaru dari OpenAI (2026)
# llm_chatgpt = ChatOpenAI(model="gpt-5.4-mini", temperature=0.3)

# Menggunakan DeepSeek-R1 via Hugging Face Hub secara gratis
llm_deepseek = HuggingFaceEndpoint(
    repo_id="deepseek-ai/DeepSeek-R1-Distill-Qwen-7B",
    task="text-generation",
    temperature=0.3
)

# --- MODEL YANG DI-KOMENTAR (Sesuai permintaan: Pro dan Gemma diparkir) ---
# llm_pro = ChatGoogleGenerativeAI(model="gemini-2.5-pro", temperature=0.3) 
# llm_gemma = ChatGoogleGenerativeAI(model="gemma-2-9b-it", temperature=0.3) 

# Pertanyaan Eksperimen
query = "Apa topik utama yang dibahas dalam dokumen ini?"

# --- PROMPT TEMPLATE ---
prompt_murni = ChatPromptTemplate.from_messages([("human", "{input}")])

system_prompt = (
    "Anda adalah asisten untuk tugas tanya jawab. "
    "Gunakan potongan konteks berikut untuk menjawab pertanyaan. "
    "Jika Anda tidak tahu jawabannya, katakan saja Anda tidak tahu.\n\n"
    "{context}"
)
prompt_rag = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# --- JALANKAN EKSPERIMEN ---

# A. Gemini 2.5 Flash (Temp 0.3)
chain_gn_murni = prompt_murni | llm_gemini_n | StrOutputParser()
jawaban_gn_murni = chain_gn_murni.invoke({"input": query})
rag_gn = ({"context": retriever | format_docs, "input": RunnablePassthrough()} | prompt_rag | llm_gemini_n | StrOutputParser())
jawaban_gn_rag = rag_gn.invoke(query)

# B. Gemini 2.5 Flash (Temp 0.7)
chain_gc_murni = prompt_murni | llm_gemini_c | StrOutputParser()
jawaban_gc_murni = chain_gc_murni.invoke({"input": query})
rag_gc = ({"context": retriever | format_docs, "input": RunnablePassthrough()} | prompt_rag | llm_gemini_c | StrOutputParser())
jawaban_gc_rag = rag_gc.invoke(query)

# C. ChatGPT (GPT-5.4-mini)
# chain_gpt_murni = prompt_murni | llm_chatgpt | StrOutputParser()
# jawaban_gpt_murni = chain_gpt_murni.invoke({"input": query})
# rag_gpt = ({"context": retriever | format_docs, "input": RunnablePassthrough()} | prompt_rag | llm_chatgpt | StrOutputParser())
# jawaban_gpt_rag = rag_gpt.invoke(query)

# D. DeepSeek (DeepSeek-R1)
chain_ds_murni = prompt_murni | llm_deepseek | StrOutputParser()
jawaban_ds_murni = chain_ds_murni.invoke({"input": query})
rag_ds = ({"context": retriever | format_docs, "input": RunnablePassthrough()} | prompt_rag | llm_deepseek | StrOutputParser())
jawaban_ds_rag = rag_ds.invoke(query)

# E. Model Dummy/Diparkir (Pro & Gemma)
jawaban_pro_murni = jawaban_gemma_murni = jawaban_gpt_murni = "Dilewati (Di-komentar)"
jawaban_pro_rag = jawaban_gemma_rag = jawaban_gpt_rag = "Dilewati (Di-komentar)"

"""
# Blok Eksekusi Gemini Pro & Gemma (Hapus triple quotes untuk mengaktifkan nanti)
chain_pro_murni = prompt_murni | llm_pro | StrOutputParser()
jawaban_pro_murni = chain_pro_murni.invoke({"input": query})
rag_pro = ({"context": retriever | format_docs, "input": RunnablePassthrough()} | prompt_rag | llm_pro | StrOutputParser())
jawaban_pro_rag = rag_pro.invoke(query)

chain_gemma_murni = prompt_murni | llm_gemma | StrOutputParser()
jawaban_gemma_murni = chain_gemma_murni.invoke({"input": query})
rag_gemma = ({"context": retriever | format_docs, "input": RunnablePassthrough()} | prompt_rag | llm_gemma | StrOutputParser())
jawaban_gemma_rag = rag_gemma.invoke(query)
"""

# 3. Cetak Output Ke Konsol Notebook
print("\n=== 🛑 HASIL BENCHMARK 6 MODEL ===")
print(f"\n⚡ GEMINI 2.5 FLASH (T 0.3) + RAG : {jawaban_gn_rag}")
print(f"🚀 GEMINI 2.5 FLASH (T 0.7) + RAG : {jawaban_gc_rag}")
print(f"🟢 ChatGPT (GPT-5.4-mini) + RAG    : {jawaban_gpt_rag}")
print(f"🐋 DEEPSEEK-R1 + RAG              : {jawaban_ds_rag}")
print(f"💤 GEMINI 2.5 PRO + RAG           : {jawaban_pro_rag}")
print(f"💎 GEMMA + RAG                    : {jawaban_gemma_rag}")

# 4. Logging Struktur Lengkap ke Weights & Biases Table
compare_table = wandb.Table(columns=["Model", "Jawaban Murni (Tanpa RAG)", "Jawaban dengan RAG"])
compare_table.add_data("Gemini 2.5 Flash (Temp 0.3)", jawaban_gn_murni, jawaban_gn_rag)
compare_table.add_data("Gemini 2.5 Flash (Temp 0.7)", jawaban_gc_murni, jawaban_gc_rag)
compare_table.add_data("ChatGPT (GPT-5.4-mini)", jawaban_gpt_murni, jawaban_gpt_rag)
compare_table.add_data("DeepSeek-R1 (Qwen)", jawaban_ds_murni, jawaban_ds_rag)
compare_table.add_data("Gemini 2.5 Pro (Idle)", jawaban_pro_murni, jawaban_pro_rag)
compare_table.add_data("Gemma 2 (Idle)", jawaban_gemma_murni, jawaban_gemma_rag)

wandb.log({"Tabel_Benchmark_6_Model_RAG": compare_table})
run.finish()

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 51.237175015s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT